# Лабораторная работа №1  
## Данные для систем ИИ: исследование, качество и подготовка

**Цель:** пройти путь от «сырого» табличного набора данных до набора, пригодного для последующего машинного обучения.

Файл данных: `ai_clients_lab1.csv`.

Выполняйте задания по порядку. Не удаляйте исходный `df`: для очистки создайте копию `clean_df`.


In [97]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 10)
pd.set_option('display.max_rows', 10)
pd.set_option("display.precision", 2)

df = pd.read_csv("ai_clients_lab1.csv")
df.head()


,CustomerId,Age,ContractMonths,MonthlySpend,UsageHours,...,Region,Plan,InternetType,AutoPay,Churn
0,C0001,43.0,15,1466.59,15.8,...,Север,Базовый,Оптика,Нет,Нет
1,C0002,25.0,25,1315.95,15.5,...,Центр,Базовый,Оптика,Нет,Нет
2,C0003,49.0,5,1901.22,15.0,...,Восток,Стандарт,DSL,Да,Нет
3,C0004,51.0,48,1159.31,14.7,...,Центр,Базовый,Оптика,Да,Нет
4,C0005,18.0,39,1079.20,20.8,...,Восток,Стандарт,Мобильный,Да,Нет


### Задание 1. Первичное знакомство
Получите размер набора данных, список столбцов, типы данных, первые и последние строки.  
Ответьте: что является объектом наблюдения? Какие признаки числовые, категориальные и идентификационные? Какой столбец можно рассматривать как будущую целевую переменную?


In [98]:
"""
Объект: Пользователь у которого есть подписка на какой-то сервис
"""
print("")
df.count()

print(f"размерность данных: {df.shape=}")

print("Названия столбцов:")
[print(column) for column in df.columns]


# print("общая информация:")
# print(df.info())

print("типы столбцов:")
print(df.dtypes)

print("первые 5 строк:")
print(df.head())
print("последние 5 строк:")
print(df.tail())



размерность данных: df.shape=(605, 11)
Названия столбцов:
CustomerId
Age
ContractMonths
MonthlySpend
UsageHours
SupportCalls
Region
Plan
InternetType
AutoPay
Churn
типы столбцов:
CustomerId            str
Age               float64
ContractMonths      int64
MonthlySpend      float64
UsageHours        float64
                   ...   
Region                str
Plan                  str
InternetType          str
AutoPay               str
Churn                 str
Length: 11, dtype: object
первые 5 строк:
  CustomerId   Age  ContractMonths  MonthlySpend  UsageHours  ...  Region  \
0      C0001  43.0              15       1466.59        15.8  ...   Север   
1      C0002  25.0              25       1315.95        15.5  ...   Центр   
2      C0003  49.0               5       1901.22        15.0  ...  Восток   
3      C0004  51.0              48       1159.31        14.7  ...   Центр   
4      C0005  18.0              39       1079.20        20.8  ...  Восток   

       Plan InternetType Auto

### Задание 2. Качество данных
Найдите:
- пропущенные значения по столбцам;
- полные дубликаты строк;
- некорректные значения возраста;
- необычно большие значения `MonthlySpend` и `UsageHours`.

Не исправляйте данные, пока не зафиксируете найденные проблемы.


Пропущенные значения по столбцам


In [99]:
df.isnull().sum()


CustomerId         0
Age               14
ContractMonths     0
MonthlySpend       7
UsageHours         0
                  ..
Region             9
Plan               0
InternetType       0
AutoPay            0
Churn              0
Length: 11, dtype: int64

Полные дубликаты строк


In [100]:
duplicates = df[df.duplicated()]
display(duplicates)
print("Кол-во полных дубликатов:", df.duplicated().sum())


,CustomerId,Age,ContractMonths,MonthlySpend,UsageHours,...,Region,Plan,InternetType,AutoPay,Churn
600,C0013,40.0,27,2335.93,54.4,...,Север,Стандарт,Оптика,Да,Нет
601,C0078,41.0,6,1328.40,38.3,...,Юг,Стандарт,Оптика,Да,Нет
602,C0145,28.0,13,1210.16,16.7,...,Север,Премиум,Оптика,Да,Нет
603,C0323,54.0,45,857.46,93.3,...,Юг,Премиум,Оптика,Нет,Нет
604,C0502,51.0,54,645.38,38.6,...,Центр,Премиум,Оптика,Да,Нет


Кол-во полных дубликатов: 5


некорректные значения возраста:


In [101]:
invalid_age = df[(df['Age'] < 0) | (df['Age'] > 100)]

if len(invalid_age) > 0:
    display(invalid_age[['CustomerId', 'Age']])


,CustomerId,Age
37,C0038,-4.0
418,C0419,132.0


Вот тут я пытался самостоятельно найти средние, и как-то по ним вычислять аномально большие значения:

средние значения:

медианные значения:


In [102]:
MonthlySpendAVG = df["MonthlySpend"].mean()
UsageHoursAVG = df["UsageHours"].mean()
display(MonthlySpendAVG)
display(UsageHoursAVG)

MonthlySpendMED = df["MonthlySpend"].median()
UsageHoursMED = df["UsageHours"].median()
display(MonthlySpendMED)
display(UsageHoursMED)


np.float64(1496.6924749163882)

np.float64(32.63933884297521)

np.float64(1465.76)

np.float64(30.0)

Моя теория выдавала слишком много значений:


In [103]:
big_MonthlySpend = df[df["MonthlySpend"] > MonthlySpendAVG]
display(big_MonthlySpend.head(n=7))


,CustomerId,Age,ContractMonths,MonthlySpend,UsageHours,...,Region,Plan,InternetType,AutoPay,Churn
2,C0003,49.0,5,1901.22,15.0,...,Восток,Стандарт,DSL,Да,Нет
5,C0006,22.0,6,1655.92,34.7,...,Восток,Базовый,Оптика,Да,Нет
7,C0008,35.0,34,1617.57,28.7,...,Центр,Стандарт,Оптика,Нет,Нет
10,C0011,50.0,16,2132.96,8.1,...,Юг,Стандарт,Оптика,Да,Нет
12,C0013,40.0,27,2335.93,54.4,...,Север,Стандарт,Оптика,Да,Нет
18,C0019,50.0,58,1620.72,14.9,...,Центр,Стандарт,Оптика,Нет,Нет
19,C0020,38.0,3,1688.90,17.0,...,Центр,Базовый,DSL,Нет,Нет


Сортировкой вроде даже что-то находилось:


In [104]:
display(df.sort_values(by="MonthlySpend", ascending=False).head(n=7))


,CustomerId,Age,ContractMonths,MonthlySpend,UsageHours,...,Region,Plan,InternetType,AutoPay,Churn
514,C0515,49.0,35,9100.00,28.3,...,Центр,Стандарт,DSL,Да,Нет
377,C0378,18.0,36,7500.00,48.2,...,Центр,Базовый,Оптика,Да,Нет
219,C0220,30.0,4,6800.00,61.4,...,Север,Стандарт,Оптика,Да,Нет
55,C0056,31.0,41,5200.00,23.8,...,Запад,Базовый,Мобильный,Нет,Нет
184,C0185,45.0,20,2657.96,31.8,...,Центр,Стандарт,Оптика,Да,Нет
445,C0446,20.0,41,2615.62,8.4,...,Запад,Базовый,DSL,Да,Нет
273,C0274,66.0,10,2506.69,19.1,...,Запад,Премиум,Мобильный,Да,Нет


Но отделить то как? Должен ведь универсальный метод быть, и я полез искать методы

Нашёл и почитал про квадрат Тьюки и межквартильный размах, и применил его:


In [105]:
# MonthlySpend
Q1 = df['MonthlySpend'].quantile(0.25)
Q3 = df['MonthlySpend'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR
unusual_spend = df[df['MonthlySpend'] > upper]
display(unusual_spend[['MonthlySpend']])


,MonthlySpend
55,5200.00
184,2657.96
219,6800.00
377,7500.00
445,2615.62
514,9100.00


Хоть это и эмпирически верное решение, но я подумал что коэффициент лучше докрутить до 1.7 минимум, потому что я не считаю что 2657.96 или 2615.62 это аномально большие значения

И да, наверное в анализе данных или ml потеря условно пары верхних значений не критична, поэтому берут 1.5, но 1.7 в данном контексте дало более точные результаты

Создал метод специальный для этого


In [106]:
def interquartile(df, by_row, factor=1.5):
    Q1 = df[by_row].quantile(0.25)
    Q3 = df[by_row].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + factor * IQR
    return df[df[by_row] <= upper]


итого:


In [107]:
display(interquartile(df=df, by_row='MonthlySpend', factor=1.7))
display(interquartile(df=df, by_row='UsageHours', factor=1.7))

,CustomerId,Age,ContractMonths,MonthlySpend,UsageHours,...,Region,Plan,InternetType,AutoPay,Churn
0,C0001,43.0,15,1466.59,15.8,...,Север,Базовый,Оптика,Нет,Нет
1,C0002,25.0,25,1315.95,15.5,...,Центр,Базовый,Оптика,Нет,Нет
2,C0003,49.0,5,1901.22,15.0,...,Восток,Стандарт,DSL,Да,Нет
3,C0004,51.0,48,1159.31,14.7,...,Центр,Базовый,Оптика,Да,Нет
4,C0005,18.0,39,1079.20,20.8,...,Восток,Стандарт,Мобильный,Да,Нет
...,...,...,...,...,...,...,...,...,...,...,...
600,C0013,40.0,27,2335.93,54.4,...,Север,Стандарт,Оптика,Да,Нет
601,C0078,41.0,6,1328.40,38.3,...,Юг,Стандарт,Оптика,Да,Нет
602,C0145,28.0,13,1210.16,16.7,...,Север,Премиум,Оптика,Да,Нет
603,C0323,54.0,45,857.46,93.3,...,Юг,Премиум,Оптика,Нет,Нет


,CustomerId,Age,ContractMonths,MonthlySpend,UsageHours,...,Region,Plan,InternetType,AutoPay,Churn
0,C0001,43.0,15,1466.59,15.8,...,Север,Базовый,Оптика,Нет,Нет
1,C0002,25.0,25,1315.95,15.5,...,Центр,Базовый,Оптика,Нет,Нет
2,C0003,49.0,5,1901.22,15.0,...,Восток,Стандарт,DSL,Да,Нет
3,C0004,51.0,48,1159.31,14.7,...,Центр,Базовый,Оптика,Да,Нет
4,C0005,18.0,39,1079.20,20.8,...,Восток,Стандарт,Мобильный,Да,Нет
...,...,...,...,...,...,...,...,...,...,...,...
599,C0600,18.0,10,1182.31,46.5,...,Юг,Стандарт,Оптика,Да,Нет
600,C0013,40.0,27,2335.93,54.4,...,Север,Стандарт,Оптика,Да,Нет
601,C0078,41.0,6,1328.40,38.3,...,Юг,Стандарт,Оптика,Да,Нет
602,C0145,28.0,13,1210.16,16.7,...,Север,Премиум,Оптика,Да,Нет


### Задание 3. Очистка
Создайте `clean_df = df.copy()` и:
1. удалите полные дубликаты;
2. некорректный возраст замените на `NaN`, затем заполните пропуски медианой;
3. пропуски `Region` заполните наиболее частой категорией;
4. пропуски `MonthlySpend` заполните медианой;
5. выбросы `MonthlySpend` определите по правилу IQR и удалите только верхние экстремальные значения;
6. для `UsageHours` определите выбросы и примите обоснованное решение: удалить, ограничить или оставить.

После очистки проверьте данные повторно.


Не стал подписывать всё и дробить с выводом

In [108]:
# Удаление дубликатов
clean_df = df.copy()

clean_df.drop_duplicates(inplace=True)

# присваивание некореетным значениям в столбце возраста пустых значений NaN из numpy
clean_df.loc[invalid_age.index, "Age"] = np.nan

# Заемна пустых значений на медиану
clean_df["Age"] = clean_df["Age"].fillna(clean_df["Age"].median())

# Вычисление самого многочисленного региона
popular_region = clean_df['Region'].value_counts().idxmax()

# Замена пропусков на этот регион
clean_df["Region"] = clean_df["Region"].fillna(popular_region)

# Заполнение медианой
clean_df["MonthlySpend"] = clean_df["MonthlySpend"].fillna(clean_df["MonthlySpend"].median())
print("после удаления выбросов MonthlySpend:", clean_df.shape)


clean_df = interquartile(clean_df, "MonthlySpend", factor=1.5)

print("до удаления выбросов MonthlySpend:", clean_df.shape)

print(clean_df["UsageHours"].describe())
print("Макс:", clean_df["UsageHours"].max())

Q1 = clean_df["UsageHours"].quantile(0.25)
Q3 = clean_df["UsageHours"].quantile(0.75)
upper = Q3 + 1.5 * (Q3 - Q1)

to_clean = (clean_df["UsageHours"] > upper).sum()
print(f"Значений выше границы: {to_clean}")

clean_df.loc[clean_df["UsageHours"] > upper, "UsageHours"] = upper

print("пропуски:", clean_df.isnull().sum().sum())

после удаления выбросов MonthlySpend: (600, 11)
до удаления выбросов MonthlySpend: (594, 11)
count    594.00
mean      32.50
std       18.55
min        3.00
25%       20.32
50%       29.95
75%       40.38
max      190.00
Name: UsageHours, dtype: float64
Макс: 190.0
Значений выше границы: 21
пропуски: 0


### Задание 4. Одномерный анализ
Постройте:
- гистограмму возраста;
- гистограмму месячных расходов;
- столбчатую диаграмму тарифных планов;
- boxplot для `MonthlySpend`.

Для каждого графика сформулируйте 1–2 наблюдения.


In [109]:
# Ваш код


### Задание 5. Связи между признаками
Исследуйте связь `Churn` минимум с тремя признаками. Обязательно включите:
- `AutoPay`;
- `Plan`;
- один числовой признак по вашему выбору.

Для категориальных признаков используйте `pd.crosstab(..., normalize='index')`. Для числового признака сравните группы через `groupby`.


In [110]:
# Ваш код


### Задание 6. Корреляции
Выберите числовые признаки и вычислите корреляционную матрицу. Визуализируйте её средствами Matplotlib (`imshow`).  
Укажите две наиболее заметные связи и объясните, почему корреляция сама по себе не доказывает причинность.


In [111]:
# Ваш код


### Задание 7. Feature engineering
Создайте два новых признака:
- `TenureGroup`: `Новый` (1–12 мес.), `Стабильный` (13–36), `Долгосрочный` (37+);
- `HighSupport`: 1, если обращений в поддержку 4 и больше, иначе 0.

Проверьте долю `Churn="Да"` в новых группах.


In [112]:
# Ваш код


### Задание 8. Подготовка X и y
Создайте:
- `y` из столбца `Churn`, преобразовав `Да/Нет` в `1/0`;
- `X` без `CustomerId` и `Churn`;
- категориальные признаки преобразуйте через `pd.get_dummies(..., drop_first=True)`.

Проверьте размерность `X` и наличие пропусков. **Модель пока не обучаем.**


In [113]:
# Ваш код


### Задание 9. Сохранение результата
Сохраните очищенный набор как `ai_clients_clean.csv`. В выводе укажите исходное и итоговое число строк.


In [114]:
# Ваш код


## Итоговый вывод
Ответьте кратко:
1. Какие проблемы качества данных были обнаружены?
2. Какие решения по очистке вы приняли и почему?
3. Какие признаки визуально/аналитически связаны с оттоком?
4. Почему нельзя по этим наблюдениям утверждать, что найденные признаки **причинно** вызывают отток?
5. Почему подготовка данных является частью разработки системы ИИ, а не отдельной «технической» процедурой?
